# Load street-level crime data

In [1]:
import pandas as pd
import sqlite3

In [2]:
##create wales_data.db

#paths/connection
db_name = "../data/wales_data.db"   
db_connection = sqlite3.connect(db_name)

months = ["2010-12"]
for y in range(2011, 2026):
    for m in range(1, 13):
        months.append(str(y) + "-" + "{0:02}".format(m))

months += ["2026-01", "2026-02"]

policeForces = ["dyfed-powys", "gwent", "north-wales", "south-wales"]


#renaming/defining which ones to keep
#dropped: lsoa name, location, reported by, falls within
column_mapping = {
    'Crime ID': 'crime_id',
    'Month': 'month',
    'Longitude': 'longitude',
    'Latitude': 'latitude',
    'LSOA code': 'lsoa_code',
    'Crime type': 'crime_type',
    'Last outcome category': 'last_outcome'
}

#start count to see progress
successful = 0

for police in policeForces:        
    for m in months:
        try:
            #takes only columns mentioned before, low_memory bc pandas warnings
            df = pd.read_csv("../data/" + m + "/" + m + "-" + police + "-street.csv", usecols=column_mapping.keys(), low_memory=False)
                
            #rename the columns as above
            df = df.rename(columns=column_mapping)
                
            #write to db
            df.to_sql("street_crimes", db_connection, if_exists="append", index=False)
            successful += 1

            #show progress status
            if successful % 50 == 0:
                print(f"processed {successful} files")
                    
        except Exception as e:
            print("file for", police, "police in", m, "not found!")



#index for based on lsoa
db_connection.execute("CREATE INDEX IF NOT EXISTS idx_street_lsoa ON street_crimes(lsoa_code);")

#index for crimes by date
db_connection.execute("CREATE INDEX IF NOT EXISTS idx_street_month ON street_crimes(month);")

db_connection.close()
print(f"{successful} files into {db_name}")
print("Finished loading street-level crime data from Wales")

processed 50 files
processed 100 files
processed 150 files
processed 200 files
processed 250 files
processed 300 files
processed 350 files
file for gwent police in 2025-09 not found!
file for gwent police in 2025-10 not found!
file for gwent police in 2025-11 not found!
file for gwent police in 2025-12 not found!
file for gwent police in 2026-01 not found!
file for gwent police in 2026-02 not found!
processed 400 files
processed 450 files
processed 500 files
processed 550 files
processed 600 files
processed 650 files
processed 700 files
726 files into ../data/wales_data.db
Finished loading street-level crime data from Wales


# Create table with lsoa demographic

In [3]:
##create lsoa_demographic

file_path = "../data/wimd-2025-index.csv"
db_connection = sqlite3.connect("../data/wales_data.db")

#define new column for easiness later
column_mapping = {
    'LSOA code': 'lsoa_code',
    'Income': 'income_score',
    'Employment': 'employment_score',
    'Education': 'education_score',
    'Health': 'health_score',
    'Housing': 'housing_score',
    'Access to Services': 'services_score',
    'Community Safety' : 'safety_score',
    'Physical Environment': 'environment_score',
}

try:
    #load only columns mentioned in column_mapping
    df = pd.read_csv(file_path, usecols=column_mapping.keys())
    
    #rename the columns
    df = df.rename(columns=column_mapping)
#just in case
except Exception as e:
    print(f"not read: {e}")

#add population data
file_path = "../data/wales_population.csv"

try:
    #load the file with population data
    df_population = pd.read_csv(file_path, low_memory=False)
    
    df_population = df_population.set_index(["LSOA 2021 Code"])
    
    df = df.set_index(["lsoa_code"])
    #Add column for total population
    df["pop"] = 0
    #Add column for children population
    df["child_pop"] = 0
    #Add column for working population
    df["working_pop"] = 0
    #Ad coulmn for old population
    df["old_pop"] = 0

    for x in df_population.index:
        #Add total population
        df.at[x, "pop"] = df_population.loc[x, "Total"]

        #Compute children population (0-15)
        children = 0
        for y in range(0, 16):
            children += df_population.loc[x, "M" + str(y)]
            children += df_population.loc[x, "F" + str(y)]
        
        df.at[x, "child_pop"] = children
        
        #Compute working population (18-66)
        working = 0
        for y in range(18, 67):
            working += df_population.loc[x, "M" + str(y)]
            working += df_population.loc[x, "F" + str(y)]
        
        df.at[x, "working_pop"] = working

        #Compute old population (60 - 90)
        old = 0
        for y in range(60, 91):
            old += df_population.loc[x, "M" + str(y)]
            old += df_population.loc[x, "F" + str(y)]
        
        df.at[x, "old_pop"] = old
                
#just in case
except Exception as e:
    print(f"not read: {e}")
    

#write to db
df = df.reset_index()
df.to_sql("lsoa_demographics", db_connection, if_exists="replace", index=False)
print(len(df))
    
#create index for faster querying
db_connection.execute("CREATE INDEX IF NOT EXISTS idx_lsoa_code ON lsoa_demographics(lsoa_code);")

print("Finished loading lsoa_demographics")
db_connection.close()

1917
Finished loading lsoa_demographics


# Create table with lsoa info

In [4]:
##create lsoa_info

file_path_lsoas = "../data/wimd-2025-index.csv"
file_path_las = "../data/wales-la-codes.csv" 
file_path_la_to_pfa = "../data/LAD-to-PFA.csv"
db_connection = sqlite3.connect("../data/wales_data.db")

#define new column for easiness later
column_mapping_lsoas = {
    'LSOA code': 'lsoa_code',
    'LSOA name': 'lsoa_name',
    'Local Authority name': 'loc_auth_name'
}

column_mapping_las = {
    'LA name': 'loc_auth_name',
    'code': 'loc_auth_code',
}
    

try:
    #load only columns mentioned in column_mapping
    df_lsoa = pd.read_csv(file_path_lsoas, usecols=column_mapping_lsoas.keys())
    df_las = pd.read_csv(file_path_las, usecols=column_mapping_las.keys())

    #rename the columns
    df_lsoa = df_lsoa.rename(columns=column_mapping_lsoas)
    df_las = df_las.rename(columns=column_mapping_las)

    df = pd.merge(df_lsoa, df_las, on="loc_auth_name")

    #add connection to police force areas and codes
    df["pfa_code"] = ""
    df["pfa_name"] = ""
    
    df_la_to_pfa = pd.read_csv(file_path_la_to_pfa)
    df_la_to_pfa = df_la_to_pfa.set_index(["LAD21CD"])

    df = df.set_index(["lsoa_code"])

    for x in df.index:
        local_authority = df.loc[x, "loc_auth_code"]
        pfa_code = df_la_to_pfa.loc[local_authority, "PFA21CD"]
        pfa_name = df_la_to_pfa.loc[local_authority, "PFA21NM"]

        df.at[x, "pfa_code"] = pfa_code
        df.at[x, "pfa_name"] = pfa_name

    df = df.reset_index()
    #write to db
    df.to_sql("lsoa_info", db_connection, if_exists="replace", index=False)
    print(len(df))
    
    #create index for faster querying
    db_connection.execute("CREATE INDEX IF NOT EXISTS idx_lsoa_code ON lsoa_info(lsoa_code);")

#just in case
except Exception as e:
    print(f"not read: {e}")

db_connection.close()

1917


# Load weather data

In [5]:
#creating pfa_weather; the same gemini-assisted code as used in 01_load_data.ipynb

db_name = "../data/wales_data.db"
file_path = "../data/weather_dataset.csv" 
db_connection = sqlite3.connect(db_name)

def clean_pfa_name(series):
    return (series.astype(str)
            .str.lower()
            .str.replace(' constabulary', '', regex=False)
            .str.replace(' police', '', regex=False)
            .str.replace(' service', '', regex=False)
            .str.replace('&', 'and', regex=False)
            .str.strip())

try:
    print("Loading weather data...")
    df = pd.read_csv(file_path)
    
    # --- FIX 1: Manual Overrides ---
    # Force the CSV to match the database's weird naming convention
    manual_fixes = {
        'City of London Police': 'London, City of'
    }
    df['policeForce'] = df['policeForce'].replace(manual_fixes)
    
    # 1. RESHAPE
    print("Reshaping weather data from Wide to Long format...")
    melted = df.melt(id_vars=['policeForce', 'weatherStation'], var_name='time_var', value_name='val')
    melted[['year', 'month_num', 'measurement']] = melted['time_var'].str.split('_', expand=True)
    melted['month'] = melted['year'] + '-' + melted['month_num'].str.zfill(2)
    weather_long = melted.pivot_table(index=['policeForce', 'month'], columns='measurement', values='val').reset_index()
    weather_long.columns.name = None 
    
    # 2. MAP WITH STRING CLEANING
    print("Fetching PFA mapping from lsoa_info...")
    mapping_query = """
    SELECT DISTINCT pfa_name, pfa_code 
    FROM lsoa_info 
    WHERE pfa_name IS NOT NULL;
    """
    pfa_mapping = pd.read_sql(mapping_query, db_connection)
    
    weather_long['match_name'] = clean_pfa_name(weather_long['policeForce'])
    pfa_mapping['match_name'] = clean_pfa_name(pfa_mapping['pfa_name'])
    
    weather_long = weather_long.merge(pfa_mapping, on='match_name', how='left')
    
    # Check for missing codes again
    missing_codes = weather_long[weather_long['pfa_code'].isna()]['policeForce'].unique()
    
    # Report only errors regarding Welsh police forces
    expected = ['Dyfed-Powys Police', 'Gwent Police', 'North Wales Police', 'South Wales Police']
    actual_errors = [m for m in missing_codes if m in expected]
    
    if len(actual_errors) > 0:
        print(f"\n[!] Still failing to match these specific forces: {actual_errors}")
    else:
        print("\nPerfect Match! All active Welsh Police Forces successfully linked. (England and NI safely ignored).")
        
    # Clean up the extra columns
    weather_long = weather_long.drop(columns=['policeForce', 'match_name'])
    
    # --- FIX 2: Drop Unmapped Regions ---
    # Since Wales and NI aren't in our LSOA database, we delete their weather data
    weather_long = weather_long.dropna(subset=['pfa_code'])
    
    # 3. WRITE TO DATABASE
    print("Writing clean table to database...")
    weather_long.to_sql("pfa_weather", db_connection, if_exists="replace", index=False)
    
    db_connection.execute("CREATE INDEX IF NOT EXISTS idx_weather_pfa ON pfa_weather(pfa_name, month);")
    db_connection.execute("CREATE INDEX IF NOT EXISTS idx_weather_pfacode ON pfa_weather(pfa_code, month);")
    
    print(f"Success! {len(weather_long)} rows written to 'pfa_weather' in {db_name}.")

except Exception as e:
    print(f"Error processing data: {e}")

finally:
    db_connection.close()



Loading weather data...
Reshaping weather data from Wide to Long format...
Fetching PFA mapping from lsoa_info...

Perfect Match! All active Welsh Police Forces successfully linked. (England and NI safely ignored).
Writing clean table to database...
Success! 784 rows written to 'pfa_weather' in ../data/wales_data.db.
